In [1]:
import polars as pl
from procompa import get_project_root
PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

In [2]:
#yeastmap complexes categorized according to best match in train/test/complex db (if complex wasnt in train/test)
# best match was detremined based on jaccard simialrity

ym_cp_matching = pl.read_csv(PRJ_ROOT / "notebooks/Dataframes/Yeast_Map/yeastmap_complex_pairs_with_scores_incl_db.csv")

In [3]:

#side pred needs to be smaller than 20,but also bigger than size_true

ym_cp_matching = ym_cp_matching.filter((pl.col("size_pred") < 20)&(pl.col("size_pred") > pl.col("size_true")))

#add pred to the yeastmap identifier, to not confude the accession ids, then get one roew per complex (dont need pairwise info)

ym_cp_matching = ym_cp_matching.with_columns(("pred_" + pl.col("predicted_complex_id")).alias("predicted_complex_id"))



In [4]:
ym_cp_matching = ym_cp_matching.group_by(
    "predicted_complex_id", maintain_order=True
).first()

# make sure compelx has at least 3 proteins
ym_cp_matching = ym_cp_matching.filter(pl.col("size_pred") >= 3)

make sure format works with my stoic pipeline

In [5]:
# rename  #Complex ac to  #Complex_ac_db and predicted_complex_id to #Complex ac add #Complex ac
# add #Complex ac Idenitifier (and stoichiometry) of molecules in complex based on predicted_complex to protid(0)|protid(0)

# 1. Rename existing columns
pipeline_ym_input= ym_cp_matching.rename(
    {
        "#Complex ac": "#Complex_ac_db",
        "predicted_complex_id": "#Complex ac",
    }
)



In [6]:
# 2. Format predicted_complex protein IDs into 'PROT_ID(0)|PROT_ID(0)' format
pipeline_ym_input = pipeline_ym_input.with_columns(
    pl.col("predicted_complex")
    .str.split(" ")
    .list.eval(pl.element() + pl.lit("(0)"))
    .list.join("|")
    .alias("Identifiers (and stoichiometry) of molecules in complex")
)

In [7]:
# save as tsv
pipeline_ym_input.write_csv( data_dir/ "Pipeline/12_Yeastmap/input_12_yeasymap.tsv", separator="\t")